# Trimind v1 — Fine-tune codys12/Qwen3-8B-BitNet

Fine-tunes the BitNet-quantized Qwen3-8B on the TriMind dataset using LoRA.

**Runtime**: T4 GPU (high-RAM) or A100 recommended
**Disk**: ~30GB free

In [ ]:
# @title 1. Clone repo and install deps
import os, sys, subprocess, gc

REPO_URL = "https://github.com/Abdulhadi446/trimind-bitnet.git"
BRANCH = "main"

if not os.path.exists("/content/trimind-bitnet"):
    !git clone -b {BRANCH} {REPO_URL} /content/trimind-bitnet

%cd /content/trimind-bitnet

!pip install -q -r training/requirements-colab.txt
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
# @title 2. Hardware check
import torch, psutil, subprocess

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
mem = psutil.virtual_memory()
print(f"RAM: {mem.total / 1e9:.1f} GB ({mem.available / 1e9:.1f} GB free)")
result = subprocess.run(['df', '-h', '/'], capture_output=True, text=True)
print(f"Disk:\n{result.stdout}")

In [ ]:
# @title 3. Login to Hugging Face
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("Set HF_TOKEN in Colab secrets (🔑 sidebar)")

In [ ]:
# @title 4. Run test (50 steps)
%cd /content/trimind-bitnet

!python training/train.py \
    --test_run \
    --output_dir ./trimind-v1-test \
    --data_dir ./data \
    --batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_length 2048 \
    --lora_r 16 \
    --lora_alpha 32 \
    --logging_steps 5

In [ ]:
# @title 5. Full training run
%cd /content/trimind-bitnet

HF_TOKEN = os.environ.get("HF_TOKEN") or userdata.get("HF_TOKEN")

!WANDB_API_KEY={os.environ.get('WANDB_API_KEY', '')} \
 HF_TOKEN={HF_TOKEN} \
 python training/train.py \
    --output_dir ./trimind-v1-final \
    --data_dir ./data \
    --batch_size 2 \
    --gradient_accumulation_steps 8 \
    --max_length 4096 \
    --learning_rate 2e-4 \
    --lora_r 16 \
    --lora_alpha 32 \
    --num_epochs 1 \
    --save_steps 200 \
    --eval_steps 200 \
    --logging_steps 10 \
    --warmup_steps 100 \
    --use_wandb true \
    --wandb_project trimind-bitnet

In [ ]:
# @title 6. Push to Hub
from huggingface_hub import HfApi, create_repo

OUTPUT_DIR = "./trimind-v1-final"
REPO_NAME = "Abdulhadi446/Trimind-v1"

api = HfApi()
create_repo(REPO_NAME, exist_ok=True, private=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_NAME,
    repo_type="model",
)
print(f"Uploaded to https://huggingface.co/{REPO_NAME}")